# Lab 03-06: Agent Framework with MLflow| | ||---|---|| **Module** | 03 -- Application Development (30% of exam) || **Estimated Questions** | ~13 questions || **UI Sections** | Experiments, Models, Serving || **Estimated Time** | 50--60 minutes || **Difficulty** | Intermediate-Advanced |---## What Is the Agent Framework and Why Does MLflow Matter?**MLflow is THE framework for logging, versioning, and deploying GenAIapplications on Databricks.** Every chain you build, every prompt you tune,every agent you create -- it all flows through MLflow.The exam heavily tests MLflow concepts:- **Experiments** -- logical groupings of runs (like folders for your work)- **Runs** -- individual attempts within an experiment (each prompt change, each model swap)- **Model Registry (Unity Catalog)** -- where you "publish" winning models for deployment- **`mlflow.pyfunc`** -- the universal wrapper that makes ANY Python code servable as an MLflow model- **Tracing** -- automatic logging of chain execution paths for debuggingThis lab bridges from **building chains** (Lab 03-01) to **deploying them** asproduction endpoints. Without MLflow, you have code in a notebook. With MLflow,you have a versioned, reproducible, deployable artifact.---## Mental Model> **Analogy:** MLflow is like a lab notebook for a scientist. Every experiment> (prompt template change, model swap, parameter adjustment) gets logged with> its parameters, metrics, and artifacts. When you find the winning experiment,> you "publish" it to the model registry (Unity Catalog) -- like publishing> your research paper. Other teams can then find your published model, review> its metrics, and deploy it to production without needing to understand your> notebook code.>> The key insight: **MLflow does not replace your code -- it wraps it.** Your> LangChain chain, your custom Python function, your prompt template -- they> all stay the same. MLflow just adds tracking, versioning, and packaging on top.---## Exam Alert -- Common Traps| Trap | Why Students Fall For It | The Truth ||---|---|---|| "Register models in the MLflow Model Registry" | Legacy docs still mention it | On Databricks, register models in **Unity Catalog (UC)**, not the legacy MLflow Model Registry. UC provides governance, ACLs, and lineage. || "`mlflow.pyfunc` is only for sklearn models" | pyfunc sounds like a traditional ML thing | `mlflow.pyfunc` wraps **ANY Python code** as an MLflow model -- including LangChain chains, custom agents, and arbitrary inference logic. || "You need separate tracking for GenAI vs ML models" | GenAI feels different from traditional ML | MLflow tracks both. Use `mlflow.langchain` for LangChain chains, `mlflow.pyfunc` for custom code. Same experiments, same registry. || "MLflow tracing requires manual instrumentation" | Tracing sounds complex | `mlflow.langchain.autolog()` enables **automatic** trace logging for LangChain. One line of code. || "Model versions are managed by MLflow" | Legacy pattern | On Databricks, model versions are managed by **Unity Catalog**. MLflow logs the model; UC versions and governs it. |---## Prerequisites- Completed Lab 03-01 (LangChain on Databricks)- Completed Lab 03-05 (Model Selection from Hub)- Understanding of LangChain chain composition (LCEL pipe syntax)---## Exam Objectives Covered- Log GenAI models and chains with MLflow- Understand MLflow experiments, runs, and artifacts- Register models in Unity Catalog- Use `mlflow.pyfunc` to wrap custom inference logic- Enable and interpret MLflow tracing for GenAI applications- Understand model signatures and their role in serving

---## Setup

In [ ]:
# ------------------------------------------------------------------
# Setup: Install dependencies
# ------------------------------------------------------------------

%pip install openai mlflow langchain langchain-community --quiet

# Restart Python to pick up the new packages
dbutils.library.restartPython()


**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

In [ ]:
# ------------------------------------------------------------------
# Setup: Initialize clients and configure MLflow
# ------------------------------------------------------------------

from openai import OpenAI
import mlflow

# OpenAI-compatible client for Databricks

# Get auth from the notebook context (works on both classic clusters and serverless)
db_host = spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=db_token,
    base_url=f"{f'https://{db_host}'}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "workspace"
SCHEMA = "genai_labs"

# Set the MLflow experiment for this lab
mlflow.set_experiment("/Users/" + spark.conf.get("spark.databricks.notebook.username", "default") + "/lab_03_06_agent_framework")

print("Client ready.  Model:", MODEL)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment:   {mlflow.get_experiment_by_name(mlflow.get_experiment(mlflow.tracking.fluent._get_experiment_id()).name).name}")


**Expected output:**```Client ready.  Model: databricks-meta-llama-3-3-70b-instructMLflow tracking URI: databricksMLflow experiment:   /Users/<your-email>/lab_03_06_agent_framework```**What just happened:** We set up an MLflow experiment -- a logical containerfor all the runs we will create in this lab. Think of it as a project folder.Every time we log parameters, metrics, or models, they go into this experiment.

---## Step 1: MLflow Basics -- Experiments, Runs, and LoggingBefore we log GenAI-specific artifacts, let's understand the MLflowfundamentals that the exam tests.### The MLflow Hierarchy```EXPERIMENT (folder)  |  +-- RUN 1 (attempt with prompt_v1, temp=0.0)  |     |-- Parameters: {"prompt_version": "v1", "temperature": 0.0}  |     |-- Metrics:    {"accuracy": 0.85, "latency_ms": 120}  |     |-- Artifacts:  model files, config files  |     +-- Tags:       {"model_type": "langchain"}  |  +-- RUN 2 (attempt with prompt_v2, temp=0.1)  |     |-- Parameters: {"prompt_version": "v2", "temperature": 0.1}  |     |-- Metrics:    {"accuracy": 0.91, "latency_ms": 135}  |     |-- Artifacts:  model files, config files  |     +-- Tags:       {"model_type": "langchain"}  |  +-- RUN 3 ...```### Key Concepts| Concept | What It Is | Exam Keyword ||---|---|---|| **Experiment** | A named collection of runs (like a project folder) | `mlflow.set_experiment()` || **Run** | A single training/inference attempt with logged data | `mlflow.start_run()` || **Parameter** | An input setting (prompt version, temperature, model name) | `mlflow.log_param()` || **Metric** | A measured outcome (accuracy, latency, cost) | `mlflow.log_metric()` || **Artifact** | A file output (model weights, config, examples) | `mlflow.log_artifact()` || **Tag** | Metadata for search and filtering | `mlflow.set_tag()` |### UI Navigation**UI -->** Left nav --> **Experiments** --> Click your experiment name --> See all runs

In [ ]:
# ------------------------------------------------------------------
# Step 1: Log parameters, metrics, and artifacts in an MLflow run
# ------------------------------------------------------------------

import mlflow
import json
import tempfile
import os

# Start a run -- everything logged inside this block belongs to this run
with mlflow.start_run(run_name="baseline_prompt_v1") as run:
    
    # --- Log parameters (inputs to your experiment) ---
    mlflow.log_param("model_name", "databricks-meta-llama-3-3-70b-instruct")
    mlflow.log_param("prompt_version", "v1")
    mlflow.log_param("temperature", 0.0)
    mlflow.log_param("max_tokens", 256)
    
    # --- Make an LLM call ---
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is MLflow in one sentence?"}
        ],
        temperature=0.0,
        max_tokens=256
    )
    answer = response.choices[0].message.content
    
    # --- Log metrics (outputs/measurements) ---
    mlflow.log_metric("response_length", len(answer))
    mlflow.log_metric("token_count", response.usage.total_tokens)
    
    # --- Log an artifact (any file) ---
    # Save the prompt template as a file and log it
    prompt_config = {
        "system_prompt": "You are a helpful assistant.",
        "user_template": "What is {topic} in one sentence?",
        "version": "v1"
    }
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(prompt_config, f, indent=2)
        temp_path = f.name
    
    mlflow.log_artifact(temp_path, artifact_path="prompts")
    os.unlink(temp_path)  # Clean up temp file
    
    # --- Set tags for filtering ---
    mlflow.set_tag("model_type", "foundation_model_api")
    mlflow.set_tag("task", "question_answering")
    
    # Print results
    run_id = run.info.run_id
    print("=" * 60)
    print("MLFLOW RUN LOGGED SUCCESSFULLY")
    print("=" * 60)
    print(f"Run ID:          {run_id}")
    print(f"Run Name:        baseline_prompt_v1")
    print(f"Experiment:      {mlflow.get_experiment(run.info.experiment_id).name}")
    print(f"Parameters:      model_name, prompt_version, temperature, max_tokens")
    print(f"Metrics:         response_length={len(answer)}, token_count={response.usage.total_tokens}")
    print(f"Artifacts:       prompts/prompt_config.json")
    print(f"\nLLM Response:    {answer}")


**Expected output:**```============================================================MLFLOW RUN LOGGED SUCCESSFULLY============================================================Run ID:          a1b2c3d4e5f6...Run Name:        baseline_prompt_v1Experiment:      /Users/<your-email>/lab_03_06_agent_frameworkParameters:      model_name, prompt_version, temperature, max_tokensMetrics:         response_length=142, token_count=58Artifacts:       prompts/prompt_config.jsonLLM Response:    MLflow is an open-source platform for managing theend-to-end machine learning lifecycle, including experiment tracking,model packaging, and deployment.```**What just happened:** We created an MLflow run and logged:- **Parameters**: The inputs we chose (model, prompt version, temperature)- **Metrics**: The measurable outcomes (response length, token count)- **Artifacts**: Supporting files (prompt config JSON)- **Tags**: Metadata for filtering runs in the UIYou can now view this run in the Experiments UI. Every parameter, metric, andartifact is permanently recorded and searchable.

---## Step 2: Logging a LangChain Chain with `mlflow.langchain.log_model()`In Step 1, we logged parameters and metrics manually. For LangChain chains,MLflow provides a specialized `mlflow.langchain.log_model()` function thatlogs the **entire chain** as a deployable model artifact.### Why This MattersWhen you log a chain with `mlflow.langchain.log_model()`:- The chain's structure is serialized (prompt template, LLM config, output parser)- Dependencies are recorded (which packages are needed to run it)- A **model signature** is inferred (input/output schema)- The logged model can be loaded and served as-is -- no code changes needed### The Key Function```pythonmlflow.langchain.log_model(    lc_model=my_chain,           # The LangChain chain object    artifact_path="chain",       # Where to store it in the run    input_example={"query": "What is MLflow?"},  # Sample input)```

In [ ]:
# ------------------------------------------------------------------
# Step 2: Build and log a LangChain chain with MLflow
# ------------------------------------------------------------------

import mlflow
from langchain_community.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a simple LangChain chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an exam preparation tutor. Give concise, accurate answers."),
    ("user", "{question}")
])

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.0,
    max_tokens=256
)

parser = StrOutputParser()

# Compose the chain using LCEL pipe syntax
chain = prompt | llm | parser

# Log the chain with MLflow
with mlflow.start_run(run_name="langchain_chain_v1") as run:
    
    # Log the chain as an MLflow model
    model_info = mlflow.langchain.log_model(
        lc_model=chain,
        artifact_path="chain",
        input_example={"question": "What is MLflow?"},
    )
    
    # Also log the parameters for this chain configuration
    mlflow.log_param("system_prompt", "exam preparation tutor")
    mlflow.log_param("model_endpoint", "databricks-meta-llama-3-3-70b-instruct")
    mlflow.log_param("temperature", 0.0)
    
    chain_run_id = run.info.run_id
    model_uri = model_info.model_uri
    
    print("=" * 60)
    print("LANGCHAIN CHAIN LOGGED TO MLFLOW")
    print("=" * 60)
    print(f"Run ID:     {chain_run_id}")
    print(f"Model URI:  {model_uri}")
    print(f"Artifact:   chain/")
    print()
    print("The chain is now stored as a deployable MLflow model.")
    print("It can be loaded with mlflow.pyfunc.load_model() or")
    print("registered to Unity Catalog for serving.")


**Expected output:**```============================================================LANGCHAIN CHAIN LOGGED TO MLFLOW============================================================Run ID:     f1e2d3c4b5a6...Model URI:  runs:/f1e2d3c4b5a6.../chainArtifact:   chain/The chain is now stored as a deployable MLflow model.It can be loaded with mlflow.pyfunc.load_model() orregistered to Unity Catalog for serving.```**What just happened:** `mlflow.langchain.log_model()` serialized the entirechain -- prompt template, LLM configuration, output parser -- into a singleMLflow model artifact. This artifact is self-contained: anyone can load itand run inference without seeing your notebook code.

---## Step 3: Loading and Testing with `mlflow.pyfunc.load_model()``mlflow.pyfunc` is the universal interface for MLflow models. Regardless ofhow a model was created -- LangChain, sklearn, PyTorch, custom code -- oncelogged, it can be loaded with `mlflow.pyfunc.load_model()` and called witha single `.predict()` method.### Why `pyfunc` Matters for the ExamThe exam tests whether you understand that `mlflow.pyfunc`:- Is NOT limited to traditional ML models- Can wrap LangChain chains, custom agents, and arbitrary Python code- Provides a **uniform predict API** regardless of the underlying framework- Is the interface used by Model Serving endpoints```python# Load ANY MLflow model -- LangChain, sklearn, PyTorch, custommodel = mlflow.pyfunc.load_model("runs:/<run_id>/chain")# Call it with a uniform interfaceresult = model.predict({"question": "What is MLflow?"})```

In [ ]:
# ------------------------------------------------------------------
# Step 3: Load the logged chain and test it
# ------------------------------------------------------------------

import mlflow

# Load the chain as a pyfunc model
# model_uri was saved in the previous cell
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Test with multiple inputs
test_questions = [
    {"question": "What is the difference between RAG and fine-tuning?"},
    {"question": "When should you use a vector search index?"},
    {"question": "What does mlflow.pyfunc do?"},
]

print("=" * 60)
print("TESTING LOADED MODEL (mlflow.pyfunc)")
print("=" * 60)

for test_input in test_questions:
    result = loaded_model.predict(test_input)
    print(f"\nQ: {test_input['question']}")
    print(f"A: {result}")
    print("-" * 60)

print()
print("KEY INSIGHT: The loaded model uses the SAME .predict() interface")
print("regardless of whether the underlying model is LangChain, sklearn,")
print("PyTorch, or custom code. This uniformity is what makes mlflow.pyfunc")
print("so powerful -- and so important for the exam.")


**Expected output:**```============================================================TESTING LOADED MODEL (mlflow.pyfunc)============================================================Q: What is the difference between RAG and fine-tuning?A: RAG retrieves relevant documents at query time and includes themin the prompt, while fine-tuning modifies the model's weights withtraining data to permanently change its behavior...------------------------------------------------------------Q: When should you use a vector search index?A: Use a vector search index when you need to find semanticallysimilar documents for a RAG pipeline...------------------------------------------------------------Q: What does mlflow.pyfunc do?A: mlflow.pyfunc provides a universal interface for loading andrunning any MLflow model with a standard .predict() method...------------------------------------------------------------KEY INSIGHT: The loaded model uses the SAME .predict() interfaceregardless of whether the underlying model is LangChain, sklearn,PyTorch, or custom code.```**What just happened:** We loaded the chain from MLflow using `mlflow.pyfunc.load_model()`and called `.predict()` on it. The key exam point: `mlflow.pyfunc` does not carewhat framework created the model. It provides a uniform prediction interface thatModel Serving endpoints use under the hood.

---## Step 4: Registering Models in Unity CatalogOn Databricks, the **Unity Catalog (UC)** is the model registry -- not thelegacy MLflow Model Registry. This is a critical exam distinction.### Unity Catalog vs Legacy MLflow Registry| Feature | Unity Catalog (Databricks) | Legacy MLflow Registry ||---|---|---|| **Governance** | Full ACLs, data lineage, audit logs | Basic permissions || **Namespace** | `catalog.schema.model_name` (3-level) | Flat model names || **Integration** | Works with all Databricks services | Standalone || **Recommended** | YES -- this is the current best practice | Legacy, being deprecated |### The Registration Pattern```python# Register a logged model to Unity Catalogmlflow.register_model(    model_uri="runs:/<run_id>/chain",    name="workspace.genai_labs.my_chain"    # catalog.schema.model_name)```### UI Navigation**UI -->** Left nav --> **Catalog** --> Navigate to `workspace.genai_labs` --> Click on the model name --> See all versions

In [ ]:
# ------------------------------------------------------------------
# Step 4: Register the model to Unity Catalog
# ------------------------------------------------------------------

import mlflow

# Set the registry URI to Unity Catalog (Databricks default)
mlflow.set_registry_uri("databricks-uc")

# Define the model name in Unity Catalog (3-level namespace)
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.exam_tutor_chain"

# Register the model
# This creates version 1 in Unity Catalog
model_version = mlflow.register_model(
    model_uri=model_uri,  # from Step 2
    name=UC_MODEL_NAME
)

print("=" * 60)
print("MODEL REGISTERED IN UNITY CATALOG")
print("=" * 60)
print(f"Model Name:    {UC_MODEL_NAME}")
print(f"Version:       {model_version.version}")
print(f"Source Run:    {model_version.source}")
print(f"Status:        {model_version.status}")
print()
print("To view in the UI:")
print(f"  Catalog --> {CATALOG} --> {SCHEMA} --> Models --> exam_tutor_chain")
print()
print("EXAM KEY POINT:")
print("  On Databricks, models are registered in UNITY CATALOG,")
print("  not the legacy MLflow Model Registry.")
print(f"  The 3-level namespace is: catalog.schema.model_name")
print(f"  Example: {UC_MODEL_NAME}")


**Expected output:**```============================================================MODEL REGISTERED IN UNITY CATALOG============================================================Model Name:    workspace.genai_labs.exam_tutor_chainVersion:       1Source Run:    runs:/f1e2d3c4b5a6.../chainStatus:        READYTo view in the UI:  Catalog --> main --> genai_labs --> Models --> exam_tutor_chainEXAM KEY POINT:  On Databricks, models are registered in UNITY CATALOG,  not the legacy MLflow Model Registry.  The 3-level namespace is: catalog.schema.model_name  Example: workspace.genai_labs.exam_tutor_chain```**What just happened:** We registered the logged chain to Unity Catalog. Thisis the "publish your paper" step from the mental model. The model is now:- **Discoverable**: Other teams can find it in the Catalog UI- **Governed**: Access controlled by Unity Catalog ACLs- **Versioned**: Version 1 is recorded; future changes create v2, v3, etc.- **Deployable**: Can be served via Model Serving with one click

---## Step 5: Versioning -- Create v2 and Compare in Experiments UIIn production, you iterate on your chain: try a different prompt template,swap the model, adjust the temperature. Each iteration is a new run.When a new run beats the previous best, you register it as a new version.### The Iteration Loop```1. Change something (prompt, model, params)2. Log a new run with mlflow.start_run()3. Evaluate: compare metrics to previous runs4. If better: register as new version in Unity Catalog5. Deploy the new version to the serving endpoint```### Comparing Runs in the UI**UI -->** Left nav --> **Experiments** --> Click your experiment -->Select multiple runs (checkboxes) --> Click **Compare**The comparison view shows parameters and metrics side-by-side, making iteasy to see what changed and whether the change improved performance.

In [ ]:
# ------------------------------------------------------------------
# Step 5: Create a v2 chain with different parameters and compare
# ------------------------------------------------------------------

import mlflow
from langchain_community.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build v2 chain -- different system prompt and temperature
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a Databricks certification exam coach. "
     "Give answers in exactly 2-3 sentences. "
     "Always mention the relevant Databricks UI section if applicable."),
    ("user", "{question}")
])

llm_v2 = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,  # Slightly higher for more natural responses
    max_tokens=256
)

chain_v2 = prompt_v2 | llm_v2 | StrOutputParser()

# Log v2 as a new run
with mlflow.start_run(run_name="langchain_chain_v2") as run_v2:
    
    model_info_v2 = mlflow.langchain.log_model(
        lc_model=chain_v2,
        artifact_path="chain",
        input_example={"question": "What is MLflow?"},
    )
    
    mlflow.log_param("system_prompt", "exam coach with UI references")
    mlflow.log_param("model_endpoint", "databricks-meta-llama-3-3-70b-instruct")
    mlflow.log_param("temperature", 0.1)
    mlflow.log_param("prompt_version", "v2")
    
    # Test and log quality metric
    test_q = "What is the difference between Foundation Model APIs and Marketplace?"
    v2_answer = chain_v2.invoke({"question": test_q})
    mlflow.log_metric("response_length", len(v2_answer))
    
    v2_run_id = run_v2.info.run_id

# Compare v1 and v2
print("=" * 70)
print("COMPARING CHAIN VERSIONS")
print("=" * 70)

print(f"\n{'Parameter':<25s} {'v1':<30s} {'v2'}")
print("-" * 70)
print(f"{'System prompt':<25s} {'exam prep tutor':<30s} {'exam coach + UI refs'}")
print(f"{'Temperature':<25s} {'0.0':<30s} {'0.1'}")
print(f"{'Prompt version':<25s} {'v1':<30s} {'v2'}")

print(f"\nv2 answer to: '{test_q}'")
print(f"  {v2_answer}")

print()
print("TO COMPARE IN THE UI:")
print("  Experiments --> lab_03_06_agent_framework")
print("  --> Check both runs --> Click 'Compare'")
print("  --> See parameters and metrics side-by-side")


**Expected output:**```======================================================================COMPARING CHAIN VERSIONS======================================================================Parameter                 v1                             v2----------------------------------------------------------------------System prompt             exam prep tutor                exam coach + UI refsTemperature               0.0                            0.1Prompt version            v1                             v2v2 answer to: 'What is the difference between Foundation Model APIs and Marketplace?'  Foundation Model APIs are pre-deployed models available instantly via  pay-per-token endpoints in the Serving UI, while Marketplace models must  be deployed to your own serving endpoint after browsing and selecting them.  You can find Foundation Model APIs under Serving and Marketplace models  under the Marketplace section in the left navigation.TO COMPARE IN THE UI:  Experiments --> lab_03_06_agent_framework  --> Check both runs --> Click 'Compare'  --> See parameters and metrics side-by-side```

In [ ]:
# ------------------------------------------------------------------
# Step 5 (continued): Register v2 to Unity Catalog
# ------------------------------------------------------------------
# If v2 is better, register it as a new version of the same model.

model_version_v2 = mlflow.register_model(
    model_uri=model_info_v2.model_uri,
    name=UC_MODEL_NAME  # Same model name --> creates version 2
)

print("=" * 60)
print("VERSION 2 REGISTERED")
print("=" * 60)
print(f"Model:     {UC_MODEL_NAME}")
print(f"Version:   {model_version_v2.version}")
print(f"Source:    {model_version_v2.source}")
print()
print("Now you have two versions in Unity Catalog:")
print("  v1: baseline exam tutor (temp=0.0)")
print("  v2: exam coach with UI references (temp=0.1)")
print()
print("In the Catalog UI, you can:")
print("  - View both versions side-by-side")
print("  - Set aliases (e.g., 'production' -> v2, 'staging' -> v1)")
print("  - Deploy either version to a serving endpoint")


**Expected output:**```============================================================VERSION 2 REGISTERED============================================================Model:     workspace.genai_labs.exam_tutor_chainVersion:   2Source:    runs:/a1b2c3d4.../chainNow you have two versions in Unity Catalog:  v1: baseline exam tutor (temp=0.0)  v2: exam coach with UI references (temp=0.1)```

---## Step 6: The MLflow Model Signature -- Why Input/Output Schema MattersThe **model signature** defines the expected input and output schema of anMLflow model. It is automatically inferred when you provide `input_example` to`log_model()`, but you can also define it explicitly.### Why Signatures Matter1. **Serving validation**: Model Serving uses the signature to validate   incoming requests. If a request does not match the schema, it is rejected   before reaching your model -- saving compute and preventing errors.2. **Documentation**: The signature tells consumers exactly what inputs your   model expects and what outputs it produces.3. **Type safety**: Prevents runtime errors from mismatched data types.### Signature Structure```pythonfrom mlflow.models import infer_signature# The signature has two parts:# - inputs:  what the model expects (e.g., {"question": "string"})# - outputs: what the model returns (e.g., "string")signature = infer_signature(    model_input={"question": "What is MLflow?"},    model_output="MLflow is an open-source platform...")```### Exam Key Point> When you serve a model via Model Serving, the signature determines the> expected request/response format for the REST API endpoint. Without a> signature, the endpoint cannot validate inputs.

In [ ]:
# ------------------------------------------------------------------
# Step 6: Examine and understand model signatures
# ------------------------------------------------------------------

import mlflow
from mlflow.models import infer_signature

# Load the model and examine its signature
loaded = mlflow.pyfunc.load_model(model_uri)

if loaded.metadata.signature:
    sig = loaded.metadata.signature
    print("=" * 60)
    print("MODEL SIGNATURE")
    print("=" * 60)
    print(f"\nInputs:")
    print(f"  {sig.inputs}")
    print(f"\nOutputs:")
    print(f"  {sig.outputs}")
else:
    print("No signature found (will be inferred from input_example)")

# --- Demonstrate explicit signature creation ---
print()
print("=" * 60)
print("CREATING AN EXPLICIT SIGNATURE")
print("=" * 60)

# For a chain that takes a question and returns a string answer:
explicit_sig = infer_signature(
    model_input={"question": "What is MLflow?"},
    model_output="MLflow is an open-source platform for managing the ML lifecycle."
)

print(f"\nInputs:  {explicit_sig.inputs}")
print(f"Outputs: {explicit_sig.outputs}")
print()
print("WHY THIS MATTERS FOR SERVING:")
print("  When this model is deployed to a serving endpoint,")
print("  the REST API will expect requests like:")
print('  POST /serving-endpoints/my-endpoint/invocations')
print('  {"inputs": {"question": "What is MLflow?"}}')
print()
print("  And return responses like:")
print('  {"predictions": "MLflow is an open-source platform..."}')
print()
print("  If the request does not match the signature schema,")
print("  the endpoint returns a 400 error BEFORE calling the model.")


**Expected output:**```============================================================MODEL SIGNATURE============================================================Inputs:  ['question': string (required)]Outputs:  [string]============================================================CREATING AN EXPLICIT SIGNATURE============================================================Inputs:  ['question': string (required)]Outputs: [string]WHY THIS MATTERS FOR SERVING:  When this model is deployed to a serving endpoint,  the REST API will expect requests like:  POST /serving-endpoints/my-endpoint/invocations  {"inputs": {"question": "What is MLflow?"}}  ...```

---## Step 7: MLflow Tracing for GenAI DebuggingMLflow tracing logs the **complete execution path** of a chain or agent:which components ran, what inputs they received, what outputs they produced,and how long each step took. This is invaluable for debugging GenAI applications.### Enabling Tracing```python# One line -- enables automatic tracing for all LangChain operationsmlflow.langchain.autolog()```### What Gets TracedFor a LangChain chain (`prompt | llm | parser`), tracing logs:1. **Prompt template rendering** -- what was the final prompt after variable substitution?2. **LLM call** -- what was sent to the model? What came back? How long did it take?3. **Output parsing** -- how was the raw response transformed?### Viewing Traces**UI -->** Left nav --> **Experiments** --> Click your experiment -->Click on a run --> **Traces** tab### Exam Key Points- `mlflow.langchain.autolog()` enables tracing with ONE line of code- On **serverless compute**, you must call `autolog()` explicitly in every session- Traces are stored in the MLflow experiment, not in a separate system- Tracing works for both LangChain and custom `mlflow.pyfunc` models

In [ ]:
# ------------------------------------------------------------------
# Step 7: Enable MLflow tracing and run a traced chain
# ------------------------------------------------------------------

import mlflow

# Enable automatic tracing for LangChain
mlflow.langchain.autolog()

# Run the chain -- tracing is now automatic
trace_result = chain_v2.invoke({
    "question": "How do I register a model in Unity Catalog?"
})

print("=" * 60)
print("TRACED CHAIN EXECUTION")
print("=" * 60)
print(f"\nResult: {trace_result}")
print()
print("A trace has been automatically logged to your experiment.")
print("To view it:")
print("  Experiments --> lab_03_06_agent_framework")
print("  --> Click the latest run --> Traces tab")
print()
print("The trace shows:")
print("  1. ChatPromptTemplate: input variables --> rendered prompt")
print("  2. ChatDatabricks: rendered prompt --> raw LLM response")
print("  3. StrOutputParser: raw response --> parsed string")
print("  4. Latency for each step")
print()
print("EXAM KEY FACTS:")
print("  - mlflow.langchain.autolog() = one line to enable tracing")
print("  - On serverless: must call autolog() in EVERY session")
print("  - Traces show the full execution path for debugging")
print("  - Traces are stored in the MLflow experiment")


**Expected output:**```============================================================TRACED CHAIN EXECUTION============================================================Result: To register a model in Unity Catalog on Databricks, use themlflow.register_model() function with the 3-level namespace format:catalog.schema.model_name...A trace has been automatically logged to your experiment.To view it:  Experiments --> lab_03_06_agent_framework  --> Click the latest run --> Traces tabThe trace shows:  1. ChatPromptTemplate: input variables --> rendered prompt  2. ChatDatabricks: rendered prompt --> raw LLM response  3. StrOutputParser: raw response --> parsed string  4. Latency for each step  ...```

---

## Step 8: LangChain Tools and AgentExecutor

So far we've built **chains** — fixed sequences of steps (prompt → LLM → parser). But what if the LLM needs to **decide** which action to take based on the user's question? That's what **agents** do.

An agent is an LLM that can:
1. **Reason** about which tool to use
2. **Call** that tool with the right arguments
3. **Observe** the result
4. **Decide** whether to call another tool or return a final answer

The three key components:

| Component | What It Does | LangChain Class |
|---|---|---|
| **Tool** | A function the agent can call (e.g., search, calculator, database lookup) | `@tool` decorator or `StructuredTool` |
| **Agent** | The LLM + reasoning logic that decides which tool to use | `create_tool_calling_agent()` |
| **AgentExecutor** | The runtime loop that runs the agent's think → act → observe cycle | `AgentExecutor` |

### How It Works

```
User Question
     │
     ▼
┌─────────────────────────┐
│    AgentExecutor         │  ← The runtime loop
│  ┌───────────────────┐  │
│  │  Agent (LLM)      │  │  ← Decides what to do
│  │  "I need to look  │  │
│  │   up the order"   │  │
│  └────────┬──────────┘  │
│           │              │
│           ▼              │
│  ┌──────────────────┐   │
│  │  Tool: order_db  │   │  ← Runs the selected tool
│  │  → {status: ...} │   │
│  └────────┬─────────┘   │
│           │              │
│           ▼              │
│  Agent sees result,      │
│  decides to answer       │
│  (or call another tool)  │
└─────────────────────────┘
     │
     ▼
Final Answer to User
```

> **Chain vs Agent:** A chain always runs the same steps in the same order. An agent *decides* which steps to run based on the input. If the user asks a simple question, the agent may skip the tools entirely. If the question needs data, the agent calls a tool first.

In [ ]:
# ------------------------------------------------------------------
# Step 8a: Define tools with the @tool decorator
# ------------------------------------------------------------------

from langchain_core.tools import tool


@tool
def get_order_status(order_id: str) -> str:
    """Look up the current status of a customer order by order ID.
    Use this when the user asks about an order status or delivery."""
    # Simulated database lookup
    orders = {
        "ORD-001": {"status": "shipped", "eta": "March 22", "carrier": "FedEx"},
        "ORD-002": {"status": "processing", "eta": "March 25", "carrier": "pending"},
        "ORD-003": {"status": "delivered", "eta": "March 18", "carrier": "UPS"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id}: status={order['status']}, ETA={order['eta']}, carrier={order['carrier']}"
    return f"Order {order_id} not found."


@tool
def calculate_refund(amount: float, days_since_purchase: int) -> str:
    """Calculate the refund amount based on the purchase price and days since purchase.
    Use this when the user asks about a refund."""
    if days_since_purchase <= 30:
        refund = amount  # Full refund
        policy = "full refund (within 30-day window)"
    elif days_since_purchase <= 60:
        refund = amount * 0.5  # 50% refund
        policy = "50% refund (31-60 day window)"
    else:
        refund = 0.0
        policy = "no refund (past 60-day window)"
    return f"Refund: ${refund:.2f} — {policy}"


@tool
def search_faq(query: str) -> str:
    """Search the FAQ knowledge base for answers to common questions.
    Use this for general questions about policies, returns, shipping, etc."""
    faqs = {
        "return": "Our return policy allows returns within 30 days for a full refund, or 60 days for 50% store credit.",
        "shipping": "Standard shipping takes 5-7 business days. Express shipping takes 2-3 business days.",
        "warranty": "All products come with a 1-year manufacturer warranty covering defects.",
    }
    for key, answer in faqs.items():
        if key in query.lower():
            return answer
    return "No FAQ found for that question. Please contact support."


# Inspect the tools
tools = [get_order_status, calculate_refund, search_faq]

print("=== Defined Tools ===")
for t in tools:
    print(f"  {t.name}: {t.description[:70]}...")
    print(f"    Args: {t.args}")
    print()

print("KEY: The @tool decorator turns a Python function into a LangChain Tool.")
print("The docstring becomes the tool description — the agent reads this to")
print("decide WHEN to use the tool. Write clear docstrings!")


**Expected output:**
```
=== Defined Tools ===
  get_order_status: Look up the current status of a customer order by order ID...
    Args: {'order_id': {'title': 'Order Id', 'type': 'string'}}

  calculate_refund: Calculate the refund amount based on the purchase price and...
    Args: {'amount': {'title': 'Amount', 'type': 'number'}, 'days_since_purchase': ...}

  search_faq: Search the FAQ knowledge base for answers to common questions...
    Args: {'query': {'title': 'Query', 'type': 'string'}}
```

**What just happened:** The `@tool` decorator converts a Python function into a LangChain `Tool` object. The function signature becomes the tool's argument schema, and the docstring becomes the description the agent uses to decide when to call this tool. This is why **clear, descriptive docstrings** are critical — the agent's reasoning depends on them.

In [ ]:
# ------------------------------------------------------------------
# Step 8b: Create an agent with create_tool_calling_agent + AgentExecutor
# ------------------------------------------------------------------

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_models import ChatDatabricks

# 1. Define the agent's prompt (must include agent_scratchpad)
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful customer support agent. Use the available tools "
     "to look up order information, calculate refunds, or search FAQs. "
     "Always explain your reasoning before giving the final answer."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # Required for tool-calling agents
])

# 2. Create the LLM (tool-calling requires a chat model)
agent_llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.0,
    max_tokens=512
)

# 3. Create the agent (LLM + tools + prompt)
agent = create_tool_calling_agent(
    llm=agent_llm,
    tools=tools,
    prompt=agent_prompt
)

# 4. Wrap in AgentExecutor (the runtime loop)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,       # Show the agent's reasoning
    max_iterations=5,   # Safety limit to prevent infinite loops
    handle_parsing_errors=True
)

print("Agent created with 3 tools:")
for t in tools:
    print(f"  - {t.name}")
print(f"\nAgentExecutor ready (max {agent_executor.max_iterations} iterations)")
print("verbose=True means you'll see the agent's reasoning process.")


In [ ]:
# ------------------------------------------------------------------
# Step 8c: Run the agent with different questions
# ------------------------------------------------------------------

# Test 1: Needs the order lookup tool
print("=" * 60)
print("TEST 1: Order status question (should use get_order_status)")
print("=" * 60)
result1 = agent_executor.invoke({"input": "What's the status of order ORD-001?"})
print(f"\nFinal Answer: {result1['output']}")


In [ ]:
# Test 2: Needs the refund calculator tool
print("=" * 60)
print("TEST 2: Refund question (should use calculate_refund)")
print("=" * 60)
result2 = agent_executor.invoke({"input": "I bought something for $150 about 45 days ago. Can I get a refund?"})
print(f"\nFinal Answer: {result2['output']}")


In [ ]:
# Test 3: Needs the FAQ search tool
print("=" * 60)
print("TEST 3: General question (should use search_faq)")
print("=" * 60)
result3 = agent_executor.invoke({"input": "How long does shipping usually take?"})
print(f"\nFinal Answer: {result3['output']}")


In [ ]:
# Test 4: Simple question — agent should answer directly without tools
print("=" * 60)
print("TEST 4: Simple greeting (should NOT use any tool)")
print("=" * 60)
result4 = agent_executor.invoke({"input": "Hi, thanks for your help!"})
print(f"\nFinal Answer: {result4['output']}")


**Expected output (with verbose=True, you see the reasoning):**
```
============================================================
TEST 1: Order status question (should use get_order_status)
============================================================
> Entering new AgentExecutor chain...
> Invoking: get_order_status with {'order_id': 'ORD-001'}
> Order ORD-001: status=shipped, ETA=March 22, carrier=FedEx

Final Answer: Your order ORD-001 has been shipped via FedEx and is
expected to arrive by March 22.

============================================================
TEST 2: Refund question (should use calculate_refund)
============================================================
> Invoking: calculate_refund with {'amount': 150.0, 'days_since_purchase': 45}
> Refund: $75.00 — 50% refund (31-60 day window)

Final Answer: Since your purchase was 45 days ago, you're eligible for
a 50% refund of $75.00 under our 31-60 day policy.

============================================================
TEST 3: General question (should use search_faq)
============================================================
> Invoking: search_faq with {'query': 'shipping'}
> Standard shipping takes 5-7 business days...

Final Answer: Standard shipping takes 5-7 business days, and express
shipping takes 2-3 business days.

============================================================
TEST 4: Simple greeting (should NOT use any tool)
============================================================
Final Answer: You're welcome! Let me know if you need anything else.
```

**What just happened:** The agent demonstrated the core behavior:
- **Test 1**: Recognized an order lookup need → called `get_order_status` → used the result to answer
- **Test 2**: Recognized a refund calculation → called `calculate_refund` with extracted parameters → explained the policy
- **Test 3**: Recognized a general question → searched the FAQ → relayed the answer
- **Test 4**: Recognized no tool was needed → answered directly

This is the **key difference between a chain and an agent**: the agent *decides* which path to take.

### Component Summary

| Component | What You Write | What It Does |
|---|---|---|
| `@tool` | A Python function + docstring | Defines a capability the agent can use |
| `create_tool_calling_agent()` | LLM + tools + prompt | Creates the reasoning engine |
| `AgentExecutor` | agent + tools + config | Runs the think→act→observe loop |
| `MessagesPlaceholder("agent_scratchpad")` | Goes in the prompt | Where the agent stores intermediate reasoning |

> **Exam tip:** If a question asks about "an LLM that decides which tool to call" or "dynamic tool selection," the answer involves `AgentExecutor` + tools. If it asks about "a fixed sequence of steps," the answer is a chain (RunnableSequence).

---## Exam Tips| # | Tip | Why It Matters ||---|---|---|| 1 | **Unity Catalog is the model registry on Databricks**, not the legacy MLflow registry. Use `mlflow.set_registry_uri("databricks-uc")` and the 3-level namespace `catalog.schema.model`. | The exam will present both options. Always pick Unity Catalog. || 2 | **`mlflow.pyfunc` wraps ANY Python code** -- LangChain, sklearn, PyTorch, custom functions. It is not limited to traditional ML. | Common trap: "pyfunc is for sklearn models only" is WRONG. || 3 | **`mlflow.langchain.autolog()`** enables automatic tracing with one line. On serverless compute, call it explicitly each session. | The exam tests whether you know the autolog pattern and the serverless caveat. || 4 | **Experiments contain runs. Runs contain params, metrics, and artifacts.** This hierarchy is fundamental to MLflow. | Know the hierarchy: Experiment --> Run --> {Params, Metrics, Artifacts, Tags}. || 5 | **Model signatures define the API contract** for serving endpoints. Without a signature, the endpoint cannot validate requests. | The exam may ask "what determines the request format for a serving endpoint?" -- the answer is the model signature. || 6 | **`mlflow.langchain.log_model()` logs the entire chain** -- prompt, LLM config, parser. `mlflow.pyfunc.log_model()` logs custom Python code. | Know which `log_model` to use for LangChain vs custom code. || 7 | **`@tool` decorator** defines agent tools from Python functions. The docstring is what the agent reads to decide when to use the tool. | Clear docstrings = better tool selection by the agent. |
| 8 | **`AgentExecutor`** is the runtime loop for agents. It manages the think → act → observe cycle. `create_tool_calling_agent()` creates the agent logic. | Know the difference: chains = fixed steps, agents = dynamic tool selection. |
---## Key Takeaways### MLflow Fundamentals- **Experiment**: A named collection of runs (project folder). Set with `mlflow.set_experiment()`.- **Run**: A single attempt with logged params, metrics, and artifacts. Created with `mlflow.start_run()`.- **Parameters**: Inputs you chose (model name, temperature, prompt version).- **Metrics**: Measured outcomes (accuracy, latency, token count).- **Artifacts**: Files (model weights, config, prompt templates).### Logging GenAI Models- `mlflow.langchain.log_model()` for LangChain chains -- serializes the entire chain.- `mlflow.pyfunc.load_model()` to load ANY MLflow model with a uniform `.predict()` interface.- `mlflow.register_model("runs:/run_id/model", "catalog.schema.model_name")` to publish to Unity Catalog.### Versioning and Comparison- Each `register_model()` call to the same name creates a new version (v1, v2, v3...).- Compare runs in the Experiments UI to see which version performs best.- Set aliases in Unity Catalog (e.g., "production" -> v2) for deployment management.### Tracing- `mlflow.langchain.autolog()` enables automatic tracing -- one line.- Traces show the full execution path: prompt rendering --> LLM call --> output parsing.- On serverless compute: must call `autolog()` explicitly in every session.---## Next Lab**Lab 03-07: Multi-Agent Systems and Genie** -- you will learn how toorchestrate multiple specialized agents and how Genie translates naturallanguage into SQL for structured data queries.